# Neur — Sequential Colab Pipeline

Run each cell in order. Every step reads from and writes to Google Drive so you can stop and resume between steps.

- **Step 0** — clone repo, install deps, mount Drive.
- **Step 1** — download Samanantar (2M pairs) + FLORES-200 to `Drive/neur/raw_data/`.
- **Step 2** — clean and split the raw data into `Drive/neur/processed_data/`.
- **Step 3** — train the sinusoidal model and evaluate on FLORES-200. Outputs in `Drive/neur/outputs/`.


## Step 0 — Setup (clone, install, mount Drive)

In [ ]:
!git clone https://github.com/thenileshmishra/AS-RoPE.git /content/neur
%cd /content/neur
!pip install -q -r requirements.txt

from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.environ['NEUR_DRIVE_ROOT'] = '/content/drive/MyDrive/neur'
sys.path.insert(0, '/content/neur')

from pipeline import paths
paths.ensure_dirs()
print(paths.summary())

## Step 1 — Download raw datasets to Google Drive

Downloads up to 2M Samanantar pairs and the full FLORES-200 devtest split.
Safe to re-run — existing files are skipped unless `--force` is passed.

In [ ]:
!python -m pipeline.step1_download --sample-size 2000000

## Step 2 — Clean and split the raw data

Reads `raw_data/samanantar/samanantar_hi_en.tsv`, applies cleaning filters
(length, length-ratio, script, dedupe), and writes deterministic train/val/test
splits to `processed_data/`.

In [ ]:
!python -m pipeline.step2_preprocess

## Step 3 — Train (sinusoidal) + evaluate on FLORES-200

Trains the GPT model with sinusoidal positional encoding, saving
checkpoints under `outputs/checkpoints/`, training logs under
`outputs/logs/`, and BLEU/chrF under `outputs/metrics/`.

In [ ]:
!python -m pipeline.step3_train \
  --num-steps 12000 \
  --batch-size 16 \
  --eval-every 1000 \
  --d-model 256 --n-layers 6 --n-heads 8 \
  --max-seq-len 128

## Verify outputs on Drive

In [ ]:
!ls -lh /content/drive/MyDrive/neur/outputs/checkpoints
!cat /content/drive/MyDrive/neur/outputs/metrics/metrics.json